In [ ]:
!pip -q install pyspark==3.5.3 delta-spark==3.2.1

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.3 which is incompatible.


In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pyspark, importlib.metadata as md

builder = (SparkSession.builder.appName("delta_lake_lab")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("Spark :", spark.version)
print("Delta :", md.version("delta-spark"))

Spark : 3.5.3
Delta : 3.2.1


In [ ]:
import os, shutil
shutil.rmtree("/content/lake", ignore_errors=True)   # 다시 실행해도 깨끗하게 시작
spark.sql("DROP TABLE IF EXISTS results")

R = "/content/lake/results"
spark.sql(f"""
  CREATE TABLE results (exam_id STRING, item_code STRING, value STRING, status STRING)
  USING DELTA LOCATION '{R}'
  TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

cols = ["exam_id", "item_code", "value", "status"]
day1 = spark.createDataFrame([
    ("E001", "HB", "13.5", "final"),     # 혈색소: 당일 확정
    ("E001", "BX", None,   "pending"),   # 조직검사: 결과 대기
], cols)
day1.write.format("delta").mode("append").save(R)

spark.read.format("delta").load(R).show()
print(sorted(os.listdir(R + "/_delta_log")))

+-------+---------+-----+-------+
|exam_id|item_code|value| status|
+-------+---------+-----+-------+
|   E001|       HB| 13.5|  final|
|   E001|       BX| NULL|pending|
+-------+---------+-----+-------+

['.00000000000000000000.json.crc', '.00000000000000000001.json.crc', '00000000000000000000.json', '00000000000000000001.json', '_commits']


In [ ]:
import json
with open(R + "/_delta_log/00000000000000000001.json") as f:
    for line in f:
        action = json.loads(line)
        name = list(action.keys())[0]
        print(name, "→", json.dumps(action[name], ensure_ascii=False)[:110])

commitInfo → {"timestamp": 1794537714420, "operation": "WRITE", "operationParameters": {"mode": "Append", "partitionBy": "[
add → {"path": "part-00000-564e37cc-08da-4ded-9be6-c43be3d25241-c000.snappy.parquet", "partitionValues": {}, "size":
add → {"path": "part-00001-7d66d251-639d-4714-8bcb-de7d1ec3c452-c000.snappy.parquet", "partitionValues": {}, "size":


In [ ]:
from delta.tables import DeltaTable

updates = spark.createDataFrame([
    ("E001", "BX", "tubular adenoma", "final"),   # 조직검사 결과 도착 → 기존 행 갱신
    ("E002", "HB", "12.1", "final"),              # 새 수검자 → 새 행 삽입
], cols)

results = DeltaTable.forPath(spark, R)
(results.alias("t")
    .merge(updates.alias("s"),
           "t.exam_id = s.exam_id AND t.item_code = s.item_code")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute())

spark.read.format("delta").load(R).orderBy("exam_id", "item_code").show()
results.history().select("version", "operation",
    "operationMetrics.numTargetRowsUpdated",
    "operationMetrics.numTargetRowsInserted").show(truncate=False)

+-------+---------+---------------+------+
|exam_id|item_code|          value|status|
+-------+---------+---------------+------+
|   E001|       BX|tubular adenoma| final|
|   E001|       HB|           13.5| final|
|   E002|       HB|           12.1| final|
+-------+---------+---------------+------+

+-------+------------+--------------------+---------------------+
|version|operation   |numTargetRowsUpdated|numTargetRowsInserted|
+-------+------------+--------------------+---------------------+
|2      |MERGE       |1                   |1                    |
|1      |WRITE       |NULL                |NULL                 |
|0      |CREATE TABLE|NULL                |NULL                 |
+-------+------------+--------------------+---------------------+



In [ ]:
changes = (spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 2)        # MERGE 커밋(버전 2)부터
    .load(R))
changes.orderBy("_commit_version", "_change_type").show(truncate=False)

+-------+---------+---------------+-------+----------------+---------------+-----------------------+
|exam_id|item_code|value          |status |_change_type    |_commit_version|_commit_timestamp      |
+-------+---------+---------------+-------+----------------+---------------+-----------------------+
|E002   |HB       |12.1           |final  |insert          |2              |2026-11-13 02:44:32.916|
|E001   |BX       |tubular adenoma|final  |update_postimage|2              |2026-11-13 02:44:32.916|
|E001   |BX       |NULL           |pending|update_preimage |2              |2026-11-13 02:44:32.916|
+-------+---------+---------------+-------+----------------+---------------+-----------------------+



In [ ]:
H = "/content/lake/results_history"

history_rows = (changes
    .withColumnRenamed("_change_type", "change_type")
    .withColumnRenamed("_commit_version", "commit_version")
    .withColumnRenamed("_commit_timestamp", "commit_timestamp"))
history_rows.write.format("delta").mode("append").save(H)
spark.sql(f"ALTER TABLE delta.`{H}` SET TBLPROPERTIES (delta.appendOnly = true)")

try:
    DeltaTable.forPath(spark, H).delete("exam_id = 'E001'")
    print("삭제됨 (예상과 다름)")
except Exception as e:
    print("삭제 차단:", str(e).splitlines()[0][:150])

spark.read.format("delta").load(H).orderBy("commit_version", "change_type").show(truncate=False)

삭제 차단: [DELTA_CANNOT_MODIFY_APPEND_ONLY] This table is configured to only allow appends. If you would like to permit updates or deletes, use 'ALTER TABLE nul
+-------+---------+---------------+-------+----------------+--------------+-----------------------+
|exam_id|item_code|value          |status |change_type     |commit_version|commit_timestamp       |
+-------+---------+---------------+-------+----------------+--------------+-----------------------+
|E002   |HB       |12.1           |final  |insert          |2             |2026-11-13 02:44:32.916|
|E001   |BX       |tubular adenoma|final  |update_postimage|2             |2026-11-13 02:44:32.916|
|E001   |BX       |NULL           |pending|update_preimage |2             |2026-11-13 02:44:32.916|
+-------+---------+---------------+-------+----------------+--------------+-----------------------+



In [ ]:
spark.read.format("delta").option("versionAsOf", 1).load(R) \
     .orderBy("exam_id", "item_code").show()

+-------+---------+-----+-------+
|exam_id|item_code|value| status|
+-------+---------+-----+-------+
|   E001|       BX| NULL|pending|
|   E001|       HB| 13.5|  final|
+-------+---------+-----+-------+



In [ ]:
for i in range(10):
    spark.createDataFrame([(f"T{i:02d}", "HB", "13.0", "final")], cols) \
         .write.format("delta").mode("append").save(R)

print(sorted(f for f in os.listdir(R + "/_delta_log") if not f.endswith(".crc")))

['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json', '00000000000000000004.json', '00000000000000000005.json', '00000000000000000006.json', '00000000000000000007.json', '00000000000000000008.json', '00000000000000000009.json', '00000000000000000010.checkpoint.parquet', '00000000000000000010.json', '00000000000000000011.json', '00000000000000000012.json', '_commits', '_last_checkpoint']


In [ ]:
for f in sorted(os.listdir(R + "/_delta_log")):
    if "checkpoint" in f:
        print(f)
print("최신 버전:", DeltaTable.forPath(spark, R).history(1).select("version").first()[0])

.00000000000000000010.checkpoint.parquet.crc
._last_checkpoint.crc
00000000000000000010.checkpoint.parquet
_last_checkpoint
최신 버전: 12
